# 07 — Negative Tests and Contingency (Week 4 / Nov 5 task)

**Task (Excel Week 4, 05-Nov):** *Test duplicate input, missing mandatory field and
interrupted stage; prioritize defects. Acceptance: ≥3 negative scenarios;
documented observed outcomes; critical defects assigned and addressed.*

**Design (baseline is never touched):** every scenario stages its own inputs in a
**separate folder** of the `workspace.landing.negative_tests` volume and runs in the
isolated `workspace.neg_*` schemas, so results are directly comparable against the
clean baseline run (notebook 05). The shared run log
`workspace.negtest.pipeline_run_log` records every scenario's stage statuses.

**Scenarios:**

| # | Scenario | Input folder | Expected |
| --- | --- | --- | --- |
| 1 | Duplicate input rows | `scenario1_duplicate_input/` | Bronze keeps all raw rows (292); Silver dedupes to 267; DQ 7/7; KPIs unchanged |
| 2 | Missing mandatory field (`site_id` column absent) | `scenario2_missing_field/` | Bronze rejects the file with a clear error (DEF-01 fix); recovery rerun succeeds |
| 3 | Interrupted stage (crash mid-Silver) | `scenario3_interrupted_stage/` | Run log shows Silver Failed + partial silver; recovery rerun is idempotent (MERGE, no doubling) |

**Defects found (see NEGATIVE_TESTS.md in the repo for the full log):**
* **DEF-01 (Critical, FIXED):** a CSV missing a mandatory column was silently ingested with positionally misaligned data (birth_year landed in enrollment_date) and only failed in Silver with a confusing date-parse error. Fix: header validation in `read_csv_all_strings` (`src/transforms.py`), pinned by `tests/test_bronze_validation.py`.
* **DEF-02 (Medium, FIXED):** the bronze watermark skipped re-ingestion when the bronze table was empty (e.g. after an out-of-band reset), so a rerun "succeeded" with 0 records and silently rebuilt empty Silver/Gold. Fix: skip only if the bronze table also has data (applied to `01_bronze`, `04_orchestrate` and this harness).

Run top-to-bottom. Scenario failures are EXPECTED — they are the point.

In [0]:
# --- Bootstrap + parameterized pipeline harness with an injectable mid-Silver
# interruption (simulates an interrupted stage) ---
import datetime
import sys

from databricks.sdk.runtime import dbutils

try:
    _nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().value()
    _root = "/" + "/".join(_nb.strip("/").split("/")[:-2])
except Exception:
    _root = "/Users/sreekanthreddy2000@gmail.com/clinical-trial-dq-pip"
sys.path.insert(0, "/Workspace" + _root)

from src.transforms import (
    BUSINESS_COLUMNS, build_site_kpi_summary, build_study_kpi_summary,
    new_rows_only, read_csv_all_strings, transform_sites, transform_studies,
    transform_subjects, transform_visits,
)
from src.dq_checks import all_passed, run_dq_checks, summarize

NEG_VOL = "/Volumes/workspace/landing/negative_tests"   # SEPARATE volume: baseline untouched
NB, NS, NG = "workspace.neg_bronze", "workspace.neg_silver", "workspace.neg_gold"
NEG_RUN_LOG = "workspace.negtest.pipeline_run_log"
DATASETS = [("studies", "studies.csv"), ("sites", "sites.csv"),
            ("subjects", "subjects.csv"), ("visits", "visits.csv")]

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.negtest")
spark.sql(f"""CREATE TABLE IF NOT EXISTS {NEG_RUN_LOG} (
    run_id STRING, stage_name STRING, start_time TIMESTAMP, end_time TIMESTAMP,
    status STRING, records_processed BIGINT, error_message STRING) USING DELTA""")

def reset_neg_schemas():
    """Drop and recreate the isolated negative-test schemas (run log is kept)."""
    for s in (NB, NS, NG):
        spark.sql(f"DROP SCHEMA IF EXISTS {s} CASCADE")
        spark.sql(f"CREATE SCHEMA {s}")

def run_pipeline(run_id, landing, silver_interrupt=None):
    """Bronze -> Silver -> Gold with logging; silver_interrupt is invoked
    mid-Silver (after studies+sites upsert) to simulate a stage interruption.
    A failing stage is logged as Failed and the run stops (pipeline aborts)."""
    def _log(stage_name, start_ts, status, records=None, error=None):
        end_ts = datetime.datetime.now(datetime.UTC)
        err = None if error is None else f"{stage_name} failed: {type(error).__name__}: {error}"
        spark.createDataFrame([(run_id, stage_name, start_ts, end_ts, status, records, err)],
            "run_id string, stage_name string, start_time timestamp, end_time timestamp, "
            "status string, records_processed bigint, error_message string"
        ).createOrReplaceTempView("_nlog_row")
        spark.sql(f"""MERGE INTO {NEG_RUN_LOG} t USING _nlog_row s
                     ON t.run_id = s.run_id AND t.stage_name = s.stage_name
                     WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *""")
        msg = f"[RUN LOG] run_id={run_id} | stage={stage_name} | status={status}"
        if records is not None:
            msg += f" | records={records}"
        if err:
            msg += f" | {err}"
        print(msg)

    def _upsert(df, schema, table, key, delete_not_matched_by_source=False):
        df.createOrReplaceTempView("v_src")
        spark.sql(f"CREATE TABLE IF NOT EXISTS {schema}.{table} USING DELTA "
                 f"AS SELECT * FROM v_src WHERE 1=0")
        extra = " WHEN NOT MATCHED BY SOURCE THEN DELETE" if delete_not_matched_by_source else ""
        spark.sql(f"""MERGE INTO {schema}.{table} t USING v_src s ON t.{key} = s.{key}
                      WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *{extra}""")
        return spark.table(f"{schema}.{table}").count()

    def stage_bronze():
        wm = spark.sql(f"""SELECT MAX(end_time) AS wm FROM {NEG_RUN_LOG}
                           WHERE stage_name='Bronze_Ingestion' AND status='Success'"""
                     ).collect()[0]["wm"]
        total = 0
        for table, fname in DATASETS:
            cols = BUSINESS_COLUMNS[table]
            spark.sql(f"CREATE TABLE IF NOT EXISTS {NB}.{table} ({', '.join(c+' STRING' for c in cols)}) USING DELTA")
            path = f"{landing}/{fname}"
            mtime = datetime.datetime.fromtimestamp(dbutils.fs.ls(path)[0].modificationTime / 1000.0, tz=datetime.UTC)
            wm_utc = (wm.replace(tzinfo=datetime.UTC) if (wm is not None and wm.tzinfo is None) else wm)
            # DEF-02 hardening: skip only if the bronze table also has data
            if wm_utc is not None and mtime <= wm_utc and spark.table(f"{NB}.{table}").count() > 0:
                print(f"  bronze.{table}: SKIP (unchanged, table has data)")
                continue
            src_df = read_csv_all_strings(spark, path, cols)   # DEF-01: validates header
            new_df = new_rows_only(src_df, spark.table(f"{NB}.{table}"), cols)
            n = new_df.count()
            if n:
                col_sql = ", ".join(cols)
                new_df.createOrReplaceTempView("v_bronze_new")
                spark.sql(f"INSERT INTO {NB}.{table} ({col_sql}) SELECT {col_sql} FROM v_bronze_new")
                total += n
            print(f"  bronze.{table}: +{n}")
        return total

    def stage_silver():
        ts = datetime.datetime.now(datetime.UTC).isoformat()
        counts = 0
        counts += _upsert(transform_studies(spark.table(f"{NB}.studies"), ts), NS, "studies", "study_id")
        counts += _upsert(transform_sites(spark.table(f"{NB}.sites"), ts), NS, "sites", "site_id")
        if silver_interrupt is not None:
            silver_interrupt()   # --- simulated interruption point ---
        valid, quar = transform_subjects(spark.table(f"{NB}.subjects"), ts)
        counts += _upsert(valid, NS, "subjects", "subject_id")
        counts += _upsert(quar, NS, "subjects_quarantine", "subject_id")
        visits, visits_q = transform_visits(spark.table(f"{NB}.visits"), spark.table(f"{NS}.subjects"), ts)
        counts += _upsert(visits, NS, "visits", "visit_id", delete_not_matched_by_source=True)
        counts += _upsert(visits_q, NS, "visits_quarantine", "visit_id")
        checks = run_dq_checks(subjects=spark.table(f"{NS}.subjects"), visits=spark.table(f"{NS}.visits"),
                                studies=spark.table(f"{NS}.studies"), sites=spark.table(f"{NS}.sites"))
        print(summarize(checks))
        if not all_passed(checks):
            raise RuntimeError("Silver DQ gate failed")
        return counts

    def stage_gold():
        ts = datetime.datetime.now(datetime.UTC).isoformat()
        n1 = _upsert(build_study_kpi_summary(spark.table(f"{NS}.studies"), spark.table(f"{NS}.subjects"),
            spark.table(f"{NS}.visits"), gold_ts=ts), NG, "study_kpi_summary", "study_id")
        n2 = _upsert(build_site_kpi_summary(spark.table(f"{NS}.sites"), spark.table(f"{NS}.subjects"),
            spark.table(f"{NS}.visits"), gold_ts=ts), NG, "site_kpi_summary", "site_id")
        return n1 + n2

    results = {}
    for stage_name, fn in [("Bronze_Ingestion", stage_bronze), ("Silver_Transformation", stage_silver), ("Gold_KPI_Analytics", stage_gold)]:
        start = datetime.datetime.now(datetime.UTC)
        _log(stage_name, start, "Running")
        try:
            records = fn()
            _log(stage_name, start, "Success", records=records)
            results[stage_name] = records
        except Exception as e:
            _log(stage_name, start, "Failed", error=e)
            results[stage_name] = f"FAILED: {type(e).__name__}: {str(e)[:200]}"
            break   # negative harness records the failure and stops (pipeline aborts)
    return results

print("Negative-test harness ready")

In [0]:
# ================= SCENARIO 1: duplicate input rows =================
# Input: subjects.csv with 12 exact duplicate rows appended (280 + 12 = 292 rows).
# Expected: bronze is append-only raw (keeps 292); Silver dedupes by subject_id
# -> 267 valid + 5 quarantined (SAME as clean baseline); DQ 7/7 PASS; gold 32.
S1 = f"{NEG_VOL}/scenario1_duplicate_input"
for f in ["studies", "sites", "visits"]:
    with open(f"/Workspace{_root}/data/{f}.csv") as fh:
        dbutils.fs.put(f"{S1}/{f}.csv", fh.read(), overwrite=True)
with open(f"/Workspace{_root}/data/subjects.csv") as fh:
    lines = fh.read().splitlines()
header, data = lines[0], lines[1:]
dups = data[:12]
dbutils.fs.put(f"{S1}/subjects.csv", "\n".join([header] + data + dups) + "\n", overwrite=True)
print(f"S1 subjects.csv: {len(data)} original + {len(dups)} duplicate rows = {len(data)+len(dups)} data rows")

reset_neg_schemas()
s1 = run_pipeline("neg-s1-duplicate", S1)

obs = {"bronze_subjects": spark.table(f"{NB}.subjects").count(),
       "silver_subjects": spark.table(f"{NS}.subjects").count(),
       "quarantine": spark.table(f"{NS}.subjects_quarantine").count(),
       "gold_rows": spark.table(f"{NG}.study_kpi_summary").count() + spark.table(f"{NG}.site_kpi_summary").count()}
print("\n--- S1 observed ---", obs)
verdict = (obs["bronze_subjects"] == 292 and obs["silver_subjects"] == 267
           and obs["quarantine"] == 5 and obs["gold_rows"] == 32)
print("S1 VERDICT:", "PASS — duplicates kept in raw bronze by design, deduped in Silver, KPIs unchanged, no defect" if verdict else "FAIL — investigate!")

In [0]:
# ================= SCENARIO 2: missing mandatory field =================
# Input: subjects.csv WITHOUT the mandatory site_id column.
# PRE-FIX observed (05-Nov, live run): bronze logged "Success" with 280 MISALIGNED
# rows (Spark's explicit schema maps columns positionally — birth_year landed in
# enrollment_date) and the pipeline only failed in Silver with a confusing
# DateTimeException ("Text '1982' could not be parsed") -> DEF-01 (Critical).
# POST-FIX expected: Bronze FAILS immediately with a clear ValueError naming the
# missing column; studies+sites (processed earlier in the loop) stay consistent;
# nothing misaligned is ingested.
S2 = f"{NEG_VOL}/scenario2_missing_field"
for f in ["studies", "sites", "visits"]:
    with open(f"/Workspace{_root}/data/{f}.csv") as fh:
        dbutils.fs.put(f"{S2}/{f}.csv", fh.read(), overwrite=True)
with open(f"/Workspace{_root}/data/subjects.csv") as fh:
    lines = fh.read().splitlines()
header, data = lines[0], lines[1:]
idx = header.split(",").index("site_id")
no_site = [",".join(v for i, v in enumerate(row.split(",")) if i != idx) for row in [header] + data]
dbutils.fs.put(f"{S2}/subjects.csv", "\n".join(no_site) + "\n", overwrite=True)
print(f"S2 subjects.csv: site_id column removed; header now: {no_site[0]}")

reset_neg_schemas()
s2 = run_pipeline("neg-s2-missing-field", S2)

err = spark.sql("""SELECT error_message FROM workspace.negtest.pipeline_run_log
                   WHERE run_id='neg-s2-missing-field' AND stage_name='Bronze_Ingestion'""").collect()[0]["error_message"]
print("\n--- S2 observed (post-fix) ---")
print("Bronze error:", err[:300], "...")
obs = ("site_id" in err and "missing=[" in err
       and spark.table(f"{NB}.subjects").count() == 0
       and spark.table(f"{NB}.studies").count() == 6)
print("S2 VERDICT:", "PASS — DEF-01 fix rejects the broken file at Bronze with a clear, actionable error (defect addressed)" if obs else "FAIL — investigate!")

In [0]:
# ---- Scenario 2 contingency: recovery rerun with the correct file restored ----
# Expected: rerun succeeds; the earlier-ingested studies/sites are NOT re-appended
# (anti-join idempotency), subjects+visits ingest cleanly, DQ 7/7, gold 32.
with open(f"/Workspace{_root}/data/subjects.csv") as fh:
    dbutils.fs.put(f"{S2}/subjects.csv", fh.read(), overwrite=True)

s2r = run_pipeline("neg-s2-recovery", S2)

obs = {"bronze_subjects": spark.table(f"{NB}.subjects").count(),
       "silver_subjects": spark.table(f"{NS}.subjects").count(),
       "quarantine": spark.table(f"{NS}.subjects_quarantine").count(),
       "gold_rows": spark.table(f"{NG}.study_kpi_summary").count() + spark.table(f"{NG}.site_kpi_summary").count()}
print("\n--- S2 recovery observed ---", obs)
verdict = (s2r.get("Silver_Transformation") == 1796 and s2r.get("Gold_KPI_Analytics") == 32
           and obs["silver_subjects"] == 267 and obs["gold_rows"] == 32)
print("S2 RECOVERY VERDICT:", "PASS — fix the file, rerun; pipeline recovers idempotently (no duplicates, DQ 7/7)" if verdict else "FAIL — investigate!")

In [0]:
# ================= SCENARIO 3: interrupted stage =================
# Input: clean baseline copies (freshly staged -> new mtimes).
# Interruption: an exception is injected mid-Silver, right after studies+sites
# have been upserted (simulates a worker loss / stage crash).
# Expected: Silver logs Failed; silver contains studies (6) + sites (26) but NO
# subjects; Gold never runs; the run log shows the abort.
S3 = f"{NEG_VOL}/scenario3_interrupted_stage"
for f in ["studies", "sites", "subjects", "visits"]:
    with open(f"/Workspace{_root}/data/{f}.csv") as fh:
        dbutils.fs.put(f"{S3}/{f}.csv", fh.read(), overwrite=True)

reset_neg_schemas()
s3 = run_pipeline("neg-s3-interrupted", S3,
                  silver_interrupt=lambda: (_ for _ in ()).throw(
                      RuntimeError("Simulated interruption: worker lost mid-Silver after studies+sites upserted")))

print("\n--- S3 observed (interrupted state) ---")
print("silver.studies:", spark.table(f"{NS}.studies").count(), "| silver.sites:", spark.table(f"{NS}.sites").count())
try:
    print("silver.subjects:", spark.table(f"{NS}.subjects").count())
except Exception:
    print("silver.subjects: MISSING (never upserted) — partial state as expected")
print("S3 results:", s3)

In [0]:
# ---- Scenario 3 contingency: plain rerun (the real production recovery route) ----
# Expected: bronze SKIPs unchanged files (data intact — correct watermark use);
# Silver re-upserts EVERYTHING via MERGE -> studies stay 6 (not 12 — MERGE
# idempotency), subjects 267 (not 534), DQ 7/7 PASS, gold rebuilt (32 rows).
s3r = run_pipeline("neg-s3-recovery", S3)

obs = {"studies": spark.table(f"{NS}.studies").count(), "subjects": spark.table(f"{NS}.subjects").count(),
       "quarantine": spark.table(f"{NS}.subjects_quarantine").count(),
       "visits": spark.table(f"{NS}.visits").count(),
       "gold_rows": spark.table(f"{NG}.study_kpi_summary").count() + spark.table(f"{NG}.site_kpi_summary").count()}
print("\n--- S3 recovery observed (idempotency evidence) ---", obs)
verdict = (obs["studies"] == 6 and obs["subjects"] == 267 and obs["quarantine"] == 5
           and obs["visits"] == 1463 and obs["gold_rows"] == 32
           and s3r.get("Silver_Transformation") == 1796 and s3r.get("Gold_KPI_Analytics") == 32)
print("S3 RECOVERY VERDICT:", "PASS — rerun is idempotent: no doubling, DQ 7/7, gold rebuilt; no defect" if verdict else "FAIL — investigate!")

# ================= Full evidence: all scenario stage statuses =================
print("\n=== NEGATIVE-TEST RUN LOG (all scenarios) ===")
display(spark.sql("""SELECT run_id, stage_name, status, records_processed, error_message
                      FROM workspace.negtest.pipeline_run_log ORDER BY start_time"""))

## Defect log — Week 4 negative testing (05-Nov-2026)

| ID | Severity | Finding | Status |
| --- | --- | --- | --- |
| DEF-01 | **Critical** | CSV missing a mandatory column was ingested by Bronze with silently misaligned data (positional mapping); failure only surfaced in Silver as a confusing date-parse error | **FIXED** — header validation in `read_csv_all_strings`; pinned by `tests/test_bronze_validation.py` (4 tests); regression 18/18 pass |
| DEF-02 | Medium | Bronze watermark skipped re-ingestion when the bronze table was empty (e.g. after out-of-band reset) → rerun "succeeded" with 0 records, silently rebuilding empty Silver/Gold | **FIXED** — skip only when bronze table also has data (`01_bronze`, `04_orchestrate`, harnesses) |
| — | None | Scenario 1 (duplicates): bronze append-only keeps raw rows by design; Silver dedupes — correct behavior | No defect |
| — | None | Scenario 3 (interrupted stage): run log records the abort; rerun is idempotent (MERGE + anti-join) — correct behavior | No defect |

**Result: 2 defects found, both assigned and addressed; zero known critical defects remain.**
Full documentation: `NEGATIVE_TESTS.md`, `KPI_DICTIONARY.md`, `OPERATIONS.md` in the repo root; regression evidence: `tests/` (18 tests) + GitHub Actions CI.